# Week 3: YOLOv8 Object Detection

Clean reproduction notebook for completed UAVs@Berkeley experiments. Adapted from the Roboflow tutorial linked in the assignment. Cells were reorganized and outputs cleared for publication; this cleaned notebook has not been run end-to-end. The missing original 50-epoch cell was reconstructed from the recorded command. Measured results are preserved under `results/` and in the README. Running all cells retrains three models. Use a Colab GPU runtime. No deployment is required.

In [ ]:
!nvidia-smi
%pip install ultralytics==8.2.103 roboflow==1.1.48 -q

These are the versions used in the recorded runs. Current Colab packages can conflict with these older dependencies; check imports before proceeding. Exact reproducibility is not guaranteed across runtimes.

In [ ]:
from pathlib import Path
from getpass import getpass
from roboflow import Roboflow
from ultralytics import YOLO
import ultralytics
import os
os.chdir("/content")
ultralytics.checks()
rf = Roboflow(api_key=getpass("Roboflow API key: "))

In [ ]:
original_dataset = rf.workspace("model-examples").project("football-players-obj-detection").version(2).download("yolov8", location="/content/datasets/football-original")

## Baseline: 25 epochs
Original dataset version 2: stretch to 640 × 640, no Roboflow augmentation. YOLO training augmentation remains enabled.

In [ ]:
!yolo task=detect mode=train model=yolov8s.pt data={original_dataset.location}/data.yaml epochs=25 imgsz=800 batch=16 seed=0 plots=True name=baseline_25

## Longer training: 50 epochs
Start again from pretrained YOLOv8s weights. This is a separate run, not a continuation of the baseline.

In [ ]:
!yolo task=detect mode=train model=yolov8s.pt data={original_dataset.location}/data.yaml epochs=50 imgsz=800 batch=16 seed=0 plots=True name=experiment_50epochs

## Resize experiment
Roboflow fork version 1: fit with black padding to 640 × 640, no added Roboflow augmentations. Requires access to this fork. If unavailable, fork the source dataset and recreate these settings, retaining split membership. Brightness ±15% was previewed but not applied.

In [ ]:
fit_dataset = rf.workspace("yedil-abdiyev").project("football-players-obj-detection-nactr").version(1).download("yolov8", location="/content/datasets/football-fit-black-640")
!yolo task=detect mode=train model=yolov8s.pt data={fit_dataset.location}/data.yaml epochs=50 imgsz=800 batch=16 seed=0 plots=True name=experiment_fit_black_640

## Final evaluation
The stretched 50-epoch model was selected using validation results before this test evaluation. Do not use test results to tune further experiments. On repeated runs, YOLO may append a number to output folders; use the actual folder for the selected run.

In [ ]:
!yolo task=detect mode=val model=/content/runs/detect/experiment_50epochs/weights/best.pt data={original_dataset.location}/data.yaml split=test imgsz=800 batch=16 plots=True name=final_test_50epochs

## Save results
Download a backup before the Colab runtime expires. Model weights and datasets are excluded from the GitHub source folder.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/week3_runs", "zip", "/content/runs/detect")
files.download("/content/week3_runs.zip")